#1. Import required libraries and dataset

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import random
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from tabulate import tabulate

In [ ]:
df = pd.read_csv('/content/data_cleaned.csv')
df = df.drop(['ap_lo', 'pulse_pressure'], axis = 1)
df

,gender,height,weight,ap_hi,cholesterol,gluc,smoke,alco,active,cardio,age_years,bp_category_encoded
0,1,0.463125,-0.873002,-1.043394,1,1,0,0,1,0,-0.412905,2
1,0,-1.107092,0.933963,0.965018,3,1,0,0,1,1,0.325344,3
2,0,0.070571,-0.715875,0.295547,3,1,0,0,0,1,-0.265255,2
3,1,0.593977,0.698272,1.634488,1,1,0,0,1,1,-0.708204,3
4,0,-1.107092,-1.344384,-1.712865,1,1,0,0,0,0,-0.855854,0
...,...,...,...,...,...,...,...,...,...,...,...,...
65681,0,0.986531,-0.244493,0.295547,1,1,0,0,1,1,0.030044,2
65682,0,0.070571,0.541144,1.634488,1,1,0,0,1,1,0.620643,2
65683,1,0.463125,0.226890,-0.373924,1,1,1,0,1,0,-0.117605,2
65684,0,-0.191132,-0.087365,0.630282,1,2,0,0,0,1,1.211242,2


In [ ]:
SEED = 4240
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#2. Separate features X and target y and define functions for metrics and threshold search

In [ ]:
X = df.drop('cardio', axis = 1)
y = df['cardio']

In [ ]:
def get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred):
    table_data = [
        ['Train', precision_score(y_train, y_train_pred), recall_score(y_train, y_train_pred), f1_score(y_train, y_train_pred)],
        ['Validation', precision_score(y_val, y_val_pred), recall_score(y_val, y_val_pred), f1_score(y_val, y_val_pred)],
        ['Test', precision_score(y_test, y_test_pred), recall_score(y_test, y_test_pred), f1_score(y_test, y_test_pred)]
    ]
    print(tabulate(table_data, headers = ['Set', 'Precision', 'Recall', 'F1-Score'], tablefmt = 'grid'))

In [ ]:
def threshold_search(threshold_range, X, y, model):
    best_precision = 0
    best_recall = 0
    best_f1 = 0
    best_threshold = 0
    for threshold in threshold_range:
        y_pred = (model.predict_proba(X)[:, 1] >= threshold).astype(int)
        precision = precision_score(y, y_pred)
        recall = recall_score(y, y_pred)
        f1 = f1_score(y, y_pred)
        if (precision >= 0.7 and recall >= 0.7 and f1 > best_f1):
            best_precision = precision
            best_recall = recall
            best_f1 = f1
            best_threshold = threshold
    return best_threshold

# 3. Train basic model

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.4, stratify = y, random_state = SEED)  # 60% train, 40% temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify = y_temp, random_state = SEED)  # 20% val, 20% test

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes = (64, 32), learning_rate_init = 0.001, alpha = 0, max_iter = 30, activation = 'relu', solver = 'adam', random_state = SEED, batch_size = 64)
mlp.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(alpha=0, batch_size=64, hidden_layer_sizes=(64, 32), max_iter=30,
              random_state=4240)

In [ ]:
y_train_pred = mlp.predict(X_train)
y_val_pred = mlp.predict(X_val)
y_test_pred = mlp.predict(X_test)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

+------------+-------------+----------+------------+
| Set        |   Precision |   Recall |   F1-Score |
+============+=============+==========+============+
| Train      |    0.728269 | 0.723464 |   0.725859 |
+------------+-------------+----------+------------+
| Validation |    0.720457 | 0.712647 |   0.716531 |
+------------+-------------+----------+------------+
| Test       |    0.722466 | 0.723374 |   0.72292  |
+------------+-------------+----------+------------+


In [ ]:
best_threshold = threshold_search(np.arange(0.3, 0.5, 0.01), X_val, y_val, mlp)
print(f'Best threshold: {best_threshold}')

y_train_pred = (mlp.predict_proba(X_train)[:, 1] >= best_threshold).astype(int)
y_val_pred = (mlp.predict_proba(X_val)[:, 1] >= best_threshold).astype(int)
y_test_pred = (mlp.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

Best threshold: 0.46000000000000013
+------------+-------------+----------+------------+
| Set        |   Precision |   Recall |   F1-Score |
+============+=============+==========+============+
| Train      |    0.706448 | 0.757868 |   0.731255 |
+------------+-------------+----------+------------+
| Validation |    0.701031 | 0.74784  |   0.723679 |
+------------+-------------+----------+------------+
| Test       |    0.701124 | 0.75432  |   0.72675  |
+------------+-------------+----------+------------+


# 4. Hyperparameter tuning with 5-fold cross validation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = SEED)  # 80% train, 20% test

In [ ]:
param_grid = {
    'alpha': [0, 0.001, 0.01],
    'learning_rate_init': [0.0001, 0.001, 0.01],
    'batch_size': [64, 128, 256]
}

mlp = MLPClassifier(random_state = SEED, max_iter = 100, hidden_layer_sizes = (64, 32), activation = 'relu', solver = 'adam')
grid_search = GridSearchCV(mlp, param_grid, cv = 5, scoring = 'f1', n_jobs = -1, verbose = 2)
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


GridSearchCV(cv=5,
             estimator=MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100,
                                     random_state=4240),
             n_jobs=-1,
             param_grid={'alpha': [0, 0.001, 0.01],
                         'batch_size': [64, 128, 256],
                         'learning_rate_init': [0.0001, 0.001, 0.01]},
             scoring='f1', verbose=2)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.4, stratify = y, random_state = SEED)  # 60% train, 40% temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify = y_temp, random_state = SEED)  # 20% val, 20% test


In [ ]:
best_model = grid_search.best_estimator_
best_parameters = grid_search.best_params_

print("Best Parameters:", best_parameters)

y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)
y_test_pred = best_model.predict(X_test)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

Best Parameters: {'alpha': 0.01, 'batch_size': 64, 'learning_rate_init': 0.0001}
+------------+-------------+----------+------------+
| Set        |   Precision |   Recall |   F1-Score |
+============+=============+==========+============+
| Train      |    0.7551   | 0.668744 |   0.709303 |
+------------+-------------+----------+------------+
| Validation |    0.748267 | 0.661273 |   0.702085 |
+------------+-------------+----------+------------+
| Test       |    0.756342 | 0.674364 |   0.713004 |
+------------+-------------+----------+------------+


In [ ]:
best_threshold = threshold_search(np.arange(0.3, 0.5, 0.01), X_val, y_val, best_model)
print(f'Best threshold: {best_threshold}')

y_train_pred = (best_model.predict_proba(X_train)[:, 1] >= best_threshold).astype(int)
y_val_pred = (best_model.predict_proba(X_val)[:, 1] >= best_threshold).astype(int)
y_test_pred = (best_model.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

Best threshold: 0.4100000000000001
+------------+-------------+----------+------------+
| Set        |   Precision |   Recall |   F1-Score |
+============+=============+==========+============+
| Train      |    0.704308 | 0.755092 |   0.728817 |
+------------+-------------+----------+------------+
| Validation |    0.704242 | 0.745954 |   0.724498 |
+------------+-------------+----------+------------+
| Test       |    0.706479 | 0.755419 |   0.73013  |
+------------+-------------+----------+------------+


# 5. Hyperparameter tuning with 10-fold cross validation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = SEED)  # 80% train, 20% test

In [ ]:
param_grid = {
    'alpha': [0, 0.001, 0.01],
    'learning_rate_init': [0.0001, 0.001, 0.01],
    'batch_size': [64, 128, 256]
}

mlp = MLPClassifier(random_state = SEED, max_iter = 100, hidden_layer_sizes = (64, 32), activation = 'relu', solver = 'adam')
grid_search = GridSearchCV(mlp, param_grid, cv = 10, scoring = 'f1', n_jobs = -1, verbose = 2)
grid_search.fit(X_train, y_train)

Fitting 10 folds for each of 27 candidates, totalling 270 fits


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.4, stratify = y, random_state = SEED)  # 60% train, 40% temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify = y_temp, random_state = SEED)  # 20% val, 20% test

In [ ]:
best_model = grid_search.best_estimator_
best_parameters = grid_search.best_params_

print("Best Parameters:", best_parameters)

y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)
y_test_pred = best_model.predict(X_test)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

In [ ]:
best_threshold = threshold_search(np.arange(0.3, 0.5, 0.01), X_val, y_val, best_model)
print(f'Best threshold: {best_threshold}')

y_train_pred = (best_model.predict_proba(X_train)[:, 1] >= best_threshold).astype(int)
y_val_pred = (best_model.predict_proba(X_val)[:, 1] >= best_threshold).astype(int)
y_test_pred = (best_model.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)